# Living Paper: Complete Workflow Guide

This notebook walks you through the entire Living Paper workflow, from preparing your data to generating reviewer-ready verification packages.

**No CLI experience required.** Just run each cell in order.

---

## How to Use This Notebook

1. **Configure your paths** in the "Configuration" section below
2. **Run cells top-to-bottom** using Shift+Enter (or the Run button)
3. **Read the explanations** before each code cell
4. **Check outputs** - the notebook will tell you what happened

---

## Configuration

**Edit these values to match your project.** This is the only section you *must* modify.

In [ ]:
# ============================================================
# CONFIGURATION - Edit these paths to match your project
# ============================================================

# Path to your claims file (JSONL format)
CLAIMS_FILE = "claims.jsonl"

# Path to your evidence file (JSONL format)  
EVIDENCE_FILE = "evidence.jsonl"

# Path to your links file (CSV format)
LINKS_FILE = "links.csv"

# Path to your entities file for redaction (YAML format)
# Set to None if you don't need redaction
ENTITIES_FILE = None  # or "entities.yaml"

# Your paper ID (used for exports)
PAPER_ID = "PAPER-001"

# Where to save the reviewer package
OUTPUT_DIR = "./reviewer_package"

print("Configuration loaded!")
print(f"  Claims:   {CLAIMS_FILE}")
print(f"  Evidence: {EVIDENCE_FILE}")
print(f"  Links:    {LINKS_FILE}")
print(f"  Entities: {ENTITIES_FILE or '(no redaction)'}")
print(f"  Paper ID: {PAPER_ID}")
print(f"  Output:   {OUTPUT_DIR}")

## Setup

This cell imports the Living Paper library and sets up the environment. Run it once at the start.

In [ ]:
import sys
import os
from pathlib import Path

# Add the living-paper directory to Python path
LP_DIR = Path("__file__").resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
if str(LP_DIR) not in sys.path:
    sys.path.insert(0, str(LP_DIR))

# Change to examples directory for relative paths
os.chdir(LP_DIR / "examples")

print(f"Working directory: {Path.cwd()}")
print(f"Living Paper dir:  {LP_DIR}")
print("Setup complete!")

---

# Phase 1: Prepare Your Data

Living Paper expects three input files:

| File | Format | Contains |
|------|--------|----------|
| `claims.jsonl` | JSON Lines | Your paper's substantive claims |
| `evidence.jsonl` | JSON Lines | Evidence items (quotes, observations, etc.) |
| `links.csv` | CSV | Connections between claims and evidence |

Let's verify your files exist and peek at their contents.

In [ ]:
import json

def check_file(filepath, description):
    """Check if a file exists and show a preview."""
    path = Path(filepath)
    if not path.exists():
        print(f"ERROR: {description} not found at {filepath}")
        return False
    
    print(f"{description}: {filepath}")
    print(f"  Size: {path.stat().st_size:,} bytes")
    
    # Show first few lines
    with open(path, 'r') as f:
        lines = f.readlines()[:3]
    print(f"  Lines: {len(open(path).readlines())}")
    print(f"  Preview:")
    for line in lines:
        print(f"    {line.strip()[:80]}..." if len(line) > 80 else f"    {line.strip()}")
    print()
    return True

print("Checking input files...\n")
all_ok = True
all_ok &= check_file(CLAIMS_FILE, "Claims file")
all_ok &= check_file(EVIDENCE_FILE, "Evidence file")
all_ok &= check_file(LINKS_FILE, "Links file")

if all_ok:
    print("All input files found!")
else:
    print("Some files are missing. Please check your configuration.")

### Understanding Your Data

Let's load and inspect your claims, evidence, and links to make sure they're structured correctly.

In [ ]:
import json
import csv

# Load claims
claims = []
with open(CLAIMS_FILE, 'r') as f:
    for line in f:
        if line.strip():
            claims.append(json.loads(line))

print(f"Loaded {len(claims)} claims")
print("\nClaim types:")
types = {}
for c in claims:
    t = c.get('claim_type', 'unknown')
    types[t] = types.get(t, 0) + 1
for t, count in types.items():
    print(f"  {t}: {count}")

print("\nSample claim:")
print(f"  ID: {claims[0]['claim_id']}")
print(f"  Type: {claims[0]['claim_type']}")
print(f"  Text: {claims[0]['text'][:100]}..." if len(claims[0]['text']) > 100 else f"  Text: {claims[0]['text']}")

In [ ]:
# Load evidence
evidence = []
with open(EVIDENCE_FILE, 'r') as f:
    for line in f:
        if line.strip():
            evidence.append(json.loads(line))

print(f"Loaded {len(evidence)} evidence items")
print("\nSensitivity tiers:")
tiers = {}
for e in evidence:
    t = e.get('sensitivity_tier', 'unknown')
    tiers[t] = tiers.get(t, 0) + 1
for t, count in tiers.items():
    print(f"  {t}: {count}")

print("\nEvidence types:")
etypes = {}
for e in evidence:
    t = e.get('evidence_type', 'unknown')
    etypes[t] = etypes.get(t, 0) + 1
for t, count in etypes.items():
    print(f"  {t}: {count}")

print("\nSample evidence:")
print(f"  ID: {evidence[0]['evidence_id']}")
print(f"  Type: {evidence[0]['evidence_type']}")
print(f"  Tier: {evidence[0]['sensitivity_tier']}")
print(f"  Summary: {evidence[0]['summary'][:100]}..." if len(evidence[0]['summary']) > 100 else f"  Summary: {evidence[0]['summary']}")

In [ ]:
# Load links
links = []
with open(LINKS_FILE, 'r') as f:
    reader = csv.DictReader(f)
    links = list(reader)

print(f"Loaded {len(links)} claim-evidence links")
print("\nRelation types:")
relations = {}
for l in links:
    r = l.get('relation', 'unknown')
    relations[r] = relations.get(r, 0) + 1
for r, count in relations.items():
    print(f"  {r}: {count}")

print("\nWeight distribution:")
weights = {}
for l in links:
    w = l.get('weight', 'unknown')
    weights[w] = weights.get(w, 0) + 1
for w, count in weights.items():
    print(f"  {w}: {count}")

---

# Phase 2: Redact Identifying Information (Optional)

If your data contains identifying information (vendor names, site locations, etc.), you should redact it before sharing with reviewers.

**Skip this section** if:
- Your data is already anonymized
- You set `ENTITIES_FILE = None` above

**Use this section** if you need to replace real names with pseudonyms.

In [ ]:
if ENTITIES_FILE:
    print(f"Redaction enabled using: {ENTITIES_FILE}")
    
    # Check if entities file exists
    if Path(ENTITIES_FILE).exists():
        print("\nEntities file found. Contents:")
        with open(ENTITIES_FILE, 'r') as f:
            print(f.read())
    else:
        print(f"\nWARNING: Entities file not found at {ENTITIES_FILE}")
        print("You can copy the example and customize it:")
        print(f"  cp ../entities_example.yaml {ENTITIES_FILE}")
else:
    print("Redaction disabled (ENTITIES_FILE = None)")
    print("\nIf you need redaction, set ENTITIES_FILE in the Configuration section.")

### Preview Redactions

Before applying redactions, let's see what would be changed.

In [ ]:
if ENTITIES_FILE and Path(ENTITIES_FILE).exists():
    import subprocess
    
    print("Previewing redactions in claims file...\n")
    result = subprocess.run(
        ['python3', str(LP_DIR / 'lp.py'), 'redact', 'check', 
         '--input', CLAIMS_FILE, '--entities', ENTITIES_FILE],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print(f"Errors: {result.stderr}")
else:
    print("Skipping redaction preview (no entities file configured)")

---

# Phase 3: Initialize and Ingest

Now we'll load your data into Living Paper's database.

This creates two SQLite databases:
- `lp_public.sqlite` - Safe to share (metadata, redacted summaries)
- `lp_private.sqlite` - Keep private (pointers to raw data)

### Step 1: Initialize the Database

This creates the database files and directory structure.

In [ ]:
import subprocess

print("Initializing Living Paper database...\n")

result = subprocess.run(
    ['python3', str(LP_DIR / 'lp.py'), 'init'],
    capture_output=True, text=True,
    cwd=str(LP_DIR)
)

print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")
else:
    print("Database initialized successfully!")

### Step 2: Ingest Your Data

This loads your claims, evidence, and links into the database.

In [ ]:
print("Ingesting data into Living Paper...\n")

# Build the command
cmd = [
    'python3', str(LP_DIR / 'lp.py'), 'ingest',
    '--claims', str(Path(CLAIMS_FILE).resolve()),
    '--evidence', str(Path(EVIDENCE_FILE).resolve()),
    '--links', str(Path(LINKS_FILE).resolve())
]

# Add entities file if configured
if ENTITIES_FILE and Path(ENTITIES_FILE).exists():
    cmd.extend(['--entities', str(Path(ENTITIES_FILE).resolve())])
    print("(with redaction)\n")

result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(LP_DIR))

print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")
else:
    print("Data ingested successfully!")

### Step 3: Validate Your Data

Run the linter to check for structural issues.

In [ ]:
print("Running validation checks...\n")

result = subprocess.run(
    ['python3', str(LP_DIR / 'lp.py'), 'lint'],
    capture_output=True, text=True,
    cwd=str(LP_DIR)
)

print(result.stdout)
if result.returncode != 0:
    print(f"Validation issues found:\n{result.stderr}")
else:
    print("All validation checks passed!")

---

# Phase 4: Pre-Review Analysis

Before generating the reviewer package, let's identify any contested or problematic claims.

This helps you address issues *before* submission.

In [ ]:
prereview_path = Path(OUTPUT_DIR) / "prereview_report.md"
prereview_path.parent.mkdir(parents=True, exist_ok=True)

print("Generating pre-review analysis...\n")

result = subprocess.run(
    ['python3', str(LP_DIR / 'lp.py'), 'prereview', '--out', str(prereview_path)],
    capture_output=True, text=True,
    cwd=str(LP_DIR)
)

print(result.stdout)

if prereview_path.exists():
    print(f"\nPre-review report saved to: {prereview_path}")
    print("\n--- Report Preview ---\n")
    with open(prereview_path, 'r') as f:
        content = f.read()
        # Show first 2000 chars
        print(content[:2000])
        if len(content) > 2000:
            print(f"\n... ({len(content) - 2000} more characters)")

---

# Phase 5: Export Reviewer Package

Finally, let's generate the verification package that reviewers will use.

This creates a self-contained folder with:
- An HTML interface for reviewing claims and evidence
- README instructions for reviewers
- Launcher scripts (Mac/Windows)

In [ ]:
output_path = Path(OUTPUT_DIR).resolve()
output_path.mkdir(parents=True, exist_ok=True)

print(f"Generating reviewer package for paper: {PAPER_ID}\n")

result = subprocess.run(
    ['python3', str(LP_DIR / 'lp.py'), 'export-package', 
     '--paper', PAPER_ID, '--out', str(output_path)],
    capture_output=True, text=True,
    cwd=str(LP_DIR)
)

print(result.stdout)
if result.stderr:
    print(f"Notes: {result.stderr}")

print(f"\nPackage contents:")
for item in sorted(output_path.iterdir()):
    size = item.stat().st_size
    print(f"  {item.name:30} {size:>10,} bytes")

### Preview the HTML Interface

In [ ]:
html_path = output_path / "review.html"

if html_path.exists():
    print(f"Reviewer interface generated at:\n  {html_path}\n")
    print("To open it:")
    print(f"  - Mac: Double-click 'Open Review.command' in {output_path}")
    print(f"  - Windows: Double-click 'Open Review.bat' in {output_path}")
    print(f"  - Or open directly: {html_path}")
else:
    print("HTML file not found. Check the export output above for errors.")

---

# Summary

You've completed the Living Paper workflow!

In [ ]:
print("=" * 60)
print("LIVING PAPER WORKFLOW COMPLETE")
print("=" * 60)
print(f"\nInputs processed:")
print(f"  Claims:   {len(claims)}")
print(f"  Evidence: {len(evidence)}")
print(f"  Links:    {len(links)}")
print(f"\nOutputs generated:")
print(f"  Reviewer package: {output_path}")
print(f"\nNext steps:")
print("  1. Review the pre-review report for contested claims")
print("  2. Test the reviewer interface yourself")
print("  3. Share the reviewer_package folder with reviewers")
print("\nQuestions? Contact Matt Beane (mattbeane@ucsb.edu)")

---

## Appendix: Additional Commands

Here are some other useful Living Paper commands you can run.

### Generate Dashboard Visualization

In [ ]:
# Generate an HTML dashboard showing claim health across your papers
dashboard_path = output_path / "dashboard.html"

result = subprocess.run(
    ['python3', str(LP_DIR / 'visualize.py'), str(dashboard_path)],
    capture_output=True, text=True,
    cwd=str(LP_DIR)
)

print(result.stdout)
if dashboard_path.exists():
    print(f"\nDashboard saved to: {dashboard_path}")

### Export JSON/Markdown (for programmatic access)

In [ ]:
# Export structured data for integration with other tools
verify_export_path = output_path / "verification_data"

result = subprocess.run(
    ['python3', str(LP_DIR / 'lp.py'), 'verify-export', '--out', str(verify_export_path)],
    capture_output=True, text=True,
    cwd=str(LP_DIR)
)

print(result.stdout)
if verify_export_path.exists():
    print(f"\nExported files:")
    for item in sorted(verify_export_path.iterdir()):
        print(f"  {item.name}")